# Task description
- Classify the speakers of given features.
- Main goal: Learn how to use transformer.
- Baselines:
  - Easy: Run sample code and know how to use transformer.
  - Medium: Know how to adjust parameters of transformer.
  - Hard: Construct [conformer](https://arxiv.org/abs/2005.08100) which is a variety of transformer. 

- Other links
  - Kaggle: [link](https://www.kaggle.com/t/859c9ca9ede14fdea841be627c412322)
  - Slide: [link](https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/hw/HW04/HW04.pdf)
  - Data: [link](https://drive.google.com/file/d/1T0RPnu-Sg5eIPwQPfYysipfcz81MnsYe/view?usp=sharing)
  - Video (Chinese): [link](https://www.youtube.com/watch?v=EPerg2UnGaI)
  - Video (English): [link](https://www.youtube.com/watch?v=Gpz6AUvCak0)
  - Solution for downloading dataset fail.: [link](https://drive.google.com/drive/folders/13T0Pa_WGgQxNkqZk781qhc5T9-zfh19e?usp=sharing)

In [1]:
import os
import json
import math
import csv
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Optimizer, AdamW
from torch.optim.lr_scheduler import LambdaLR

from tqdm import tqdm
from tqdm.notebook import tqdm as notebook_tqdm
from pathlib import Path

# Download dataset
- Please follow [here](https://drive.google.com/drive/folders/13T0Pa_WGgQxNkqZk781qhc5T9-zfh19e?usp=sharing) to download data
- Data is [here](https://drive.google.com/file/d/1gaFy8RaQVUEXo2n0peCBR5gYKCB-mNHc/view?usp=sharing)

In [2]:
!gdown --id 'paste your own data download link' --output Dataset.zip
!unzip Dataset.zip

usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --id
unzip:  cannot find or open Dataset.zip, Dataset.zip.zip or Dataset.zip.ZIP.


# Data

## Dataset
- Original dataset is [Voxceleb1](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/).
- The [license](https://creativecommons.org/licenses/by/4.0/) and [complete version](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/files/license.txt) of Voxceleb1.
- We randomly select 600 speakers from Voxceleb1.
- Then preprocess the raw waveforms into mel-spectrograms.

- Args:
  - data_dir: The path to the data directory.
  - metadata_path: The path to the metadata.
  - segment_len: The length of audio segment for training. 
- The architecture of data directory \\
  - data directory \\
  |---- metadata.json \\
  |---- testdata.json \\
  |---- mapping.json \\
  |---- uttr-{random string}.pt \\

- The information in metadata
  - "n_mels": The dimention of mel-spectrogram.
  - "speakers": A dictionary. 
    - Key: speaker ids.
    - value: "feature_path" and "mel_len"


For efficiency, we segment the mel-spectrograms into segments in the traing step.

In [3]:
class myDataset(Dataset):
    def __init__(self, data_dir, segment_len=128):
        self.data_dir = data_dir
        self.segment_len = segment_len

        # 加载说话人名到 ID 的映射
        mapping_path = Path(data_dir) / "mapping.json"
        mapping = json.load(mapping_path.open())
        self.speaker2id = mapping["speaker2id"]

        # 加载训练数据的元信息
        metadata_path = Path(data_dir) / "metadata.json"
        metadata = json.load(open(metadata_path))["speakers"]

        # 获取说话人总数
        self.speaker_num = len(metadata.keys())
        self.data = []
        for speaker in metadata.keys():
            for utterances in metadata[speaker]:
                self.data.append([utterances["feature_path"], self.speaker2id[speaker]])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        feat_path, speaker = self.data[index]
        # 加载预处理好的梅尔频谱图
        mel = torch.load(os.path.join(self.data_dir, feat_path))

        # 将梅尔频谱裁剪为 segment_len 帧长度的片段
        if len(mel) > self.segment_len:
            # 随机选取裁剪起点
            start = random.randint(0, len(mel) - self.segment_len)
            # 截取 segment_len 帧
            mel = torch.FloatTensor(mel[start : start + self.segment_len])
        else:
            mel = torch.FloatTensor(mel)
        # 将说话人 ID 转为 long 类型，用于后续计算损失
        speaker = torch.FloatTensor([speaker]).long()
        return mel, speaker

    def get_speaker_number(self):
        return self.speaker_num

## Dataloader
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.


In [4]:
def collate_batch(batch):
    """拼接一个 batch 的数据"""
    mel, speaker = zip(*batch)
    # 同一 batch 内序列长度不同，需要用 pad_sequence 补齐到最长长度
    mel = pad_sequence(
        mel, batch_first=True, padding_value=-20
    )  # -20 是 log(10^(-20))，一个极小的值，不影响模型学习
    # mel 形状: (batch size, length, 40)
    return mel, torch.FloatTensor(speaker).long()


def get_dataloader(data_dir, batch_size, n_workers):
    """构建数据加载器"""
    dataset = myDataset(data_dir)
    speaker_num = dataset.get_speaker_number()
    # 按 9:1 划分为训练集和验证集
    trainlen = int(0.9 * len(dataset))
    lengths = [trainlen, len(dataset) - trainlen]
    trainset, validset = random_split(dataset, lengths)

    _pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=n_workers,
        pin_memory=_pin_memory,
        collate_fn=collate_batch,
    )
    valid_loader = DataLoader(
        validset,
        batch_size=batch_size,
        num_workers=n_workers,
        drop_last=True,
        pin_memory=_pin_memory,
        collate_fn=collate_batch,
    )

    return train_loader, valid_loader, speaker_num

# 模型
- TransformerEncoderLayer:
  - 基于 [Attention Is All You Need](https://arxiv.org/abs/1706.03762) 的标准 Transformer 编码器层
  - 参数:
    - d_model: 输入特征的期望维度（必需）

    - nhead: 多头注意力的头数（必需）

    - dim_feedforward: 前馈网络中间层的维度（默认=2048）

    - dropout: dropout 比率（默认=0.1）

    - activation: 中间层的激活函数，relu 或 gelu（默认=relu）

- TransformerEncoder:
  - TransformerEncoder 是 N 个 Transformer 编码器层的堆叠
  - 参数:
    - encoder_layer: TransformerEncoderLayer() 类的一个实例（必需）

    - num_layers: 编码器中子编码器层的数量（必需）

    - norm: 层归一化组件（可选）

In [ ]:
class ConformerConvModule(nn.Module):
    """Conformer 卷积模块: LayerNorm → Pointwise → GLU → Depthwise → BN → Swish → Pointwise → Dropout"""

    def __init__(self, d_model=80, kernel_size=31, dropout=0.3):
        super().__init__()
        self.ln = nn.LayerNorm(normalized_shape=d_model)
        self.pointwise1 = nn.Conv1d(
            in_channels=d_model, out_channels=2 * d_model, kernel_size=1
        )
        self.depthwise = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=kernel_size,
            padding=(kernel_size - 1) // 2,
            groups=d_model,
        )
        self.bn = nn.BatchNorm1d(num_features=d_model)
        self.pointwise2 = nn.Conv1d(
            in_channels=d_model, out_channels=d_model, kernel_size=1
        )
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        # x: (batch, length, d_model)
        normed = self.ln(x)
        transposed = normed.transpose(1, 2)
        expanded = self.pointwise1(transposed)
        gated = F.glu(expanded, dim=1)
        local_feat = self.depthwise(gated)
        normalized = self.bn(local_feat)
        activated = F.silu(normalized)
        projected = self.pointwise2(activated)
        conv_out = self.dropout(projected)
        return conv_out.transpose(1, 2)


class ConformerFeedForward(nn.Module):
    """半残差前馈网络: x + 0.5 * FFN(LayerNorm(x))"""

    def __init__(self, d_model=80, dim_feedforward=256, dropout=0.3):
        super().__init__()
        self.ln = nn.LayerNorm(normalized_shape=d_model)
        self.linear1 = nn.Linear(in_features=d_model, out_features=dim_feedforward)
        self.linear2 = nn.Linear(in_features=dim_feedforward, out_features=d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        normed = self.ln(x)
        expanded = self.linear1(normed)
        activated = F.silu(expanded)
        dropped = self.dropout(activated)
        projected = self.linear2(dropped)
        ffn_out = self.dropout(projected)
        return x + 0.5 * ffn_out


class ConformerLayer(nn.Module):
    """
    Conformer 编码器层 (https://arxiv.org/abs/2005.08100)
    结构: FFN → MHSA → Conv → FFN → LayerNorm
    """

    def __init__(
        self, d_model=80, nhead=2, dim_feedforward=256, conv_kernel_size=31, dropout=0.3
    ):
        super().__init__()
        self.ffn1 = ConformerFeedForward(
            d_model=d_model,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        self.mhsa_ln = nn.LayerNorm(normalized_shape=d_model)
        self.mhsa = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=False,
        )
        self.mhsa_dropout = nn.Dropout(p=dropout)
        self.conv = ConformerConvModule(
            d_model=d_model,
            kernel_size=conv_kernel_size,
            dropout=dropout,
        )
        self.ffn2 = ConformerFeedForward(
            d_model=d_model,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        self.final_ln = nn.LayerNorm(normalized_shape=d_model)

    def forward(self, x):
        # x: (length, batch, d_model)
        after_ffn1 = self.ffn1(x.transpose(0, 1)).transpose(0, 1)
        attn_input = self.mhsa_ln(after_ffn1)
        attn_output, _ = self.mhsa(attn_input, attn_input, attn_input)
        after_attn = after_ffn1 + self.mhsa_dropout(attn_output)
        conv_input = after_attn.transpose(0, 1)
        conv_output = self.conv(conv_input)
        after_conv = after_attn + conv_output.transpose(0, 1)
        after_ffn2 = self.ffn2(after_conv.transpose(0, 1)).transpose(0, 1)
        return self.final_ln(after_ffn2)


class ConformerEncoder(nn.Module):
    """堆叠 N 层 ConformerLayer"""

    def __init__(
        self,
        d_model=80,
        num_layers=2,
        nhead=2,
        dim_feedforward=256,
        conv_kernel_size=31,
        dropout=0.3,
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                ConformerLayer(
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    conv_kernel_size=conv_kernel_size,
                    dropout=dropout,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class Classifier(nn.Module):
    def __init__(self, d_model=80, n_spks=600, dropout=0.3):
        super().__init__()
        self.prenet = nn.Linear(in_features=40, out_features=d_model)
        self.encoder = ConformerEncoder(
            d_model=d_model,
            num_layers=2,
            nhead=2,
            dim_feedforward=256,
            dropout=dropout,
        )
        self.pred_layer = nn.Sequential(
            nn.Linear(in_features=d_model, out_features=d_model),
            nn.ReLU(),
            nn.Linear(in_features=d_model, out_features=n_spks),
        )

    def forward(self, mels):
        # mels: (batch, length, 40)
        projected = self.prenet(mels)
        seq_first = projected.permute(1, 0, 2)
        encoded = self.encoder(seq_first)
        batch_first = encoded.transpose(0, 1)
        pooled = batch_first.mean(dim=1)
        logits = self.pred_layer(pooled)
        return logits

# Learning rate schedule
- For transformer architecture, the design of learning rate schedule is different from that of CNN.
- Previous works show that the warmup of learning rate is useful for training models with transformer architectures.
- The warmup schedule
  - Set learning rate to 0 in the beginning.
  - The learning rate increases linearly from 0 to initial learning rate during warmup period.

In [6]:
def get_cosine_schedule_with_warmup(
  optimizer: Optimizer,
  num_warmup_steps: int,
  num_training_steps: int,
  num_cycles: float = 0.5,
  last_epoch: int = -1,
):
  """
  Create a schedule with a learning rate that decreases following the values of the cosine function between the
  initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
  initial lr set in the optimizer.

  Args:
    optimizer (:class:`~torch.optim.Optimizer`):
      The optimizer for which to schedule the learning rate.
    num_warmup_steps (:obj:`int`):
      The number of steps for the warmup phase.
    num_training_steps (:obj:`int`):
      The total number of training steps.
    num_cycles (:obj:`float`, `optional`, defaults to 0.5):
      The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
      following a half-cosine).
    last_epoch (:obj:`int`, `optional`, defaults to -1):
      The index of the last epoch when resuming training.

  Return:
    :obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
  """

  def lr_lambda(current_step):
    # Warmup
    if current_step < num_warmup_steps:
      return float(current_step) / float(max(1, num_warmup_steps))
    # decadence
    progress = float(current_step - num_warmup_steps) / float(
      max(1, num_training_steps - num_warmup_steps)
    )
    return max(
      0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
    )

  return LambdaLR(optimizer, lr_lambda, last_epoch)

# Model Function
- Model forward function.

In [7]:
def model_fn(batch, model, criterion, device):
  """Forward a batch through the model."""

  mels, labels = batch
  mels = mels.to(device)
  labels = labels.to(device)

  outs = model(mels)

  loss = criterion(outs, labels)

  # Get the speaker id with highest probability.
  preds = outs.argmax(1)
  # Compute accuracy.
  accuracy = torch.mean((preds == labels).float())

  return loss, accuracy

# Validate
- Calculate accuracy of the validation set.

In [8]:
def valid(dataloader, model, criterion, device): 
  """Validate on validation set."""

  model.eval()
  running_loss = 0.0
  running_accuracy = 0.0
  pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit=" uttr")

  for i, batch in enumerate(dataloader):
    with torch.no_grad():
      loss, accuracy = model_fn(batch, model, criterion, device)
      running_loss += loss.item()
      running_accuracy += accuracy.item()

    pbar.update(dataloader.batch_size)
    pbar.set_postfix(
      loss=f"{running_loss / (i+1):.2f}",
      accuracy=f"{running_accuracy / (i+1):.2f}",
    )

  pbar.close()
  model.train()

  return running_accuracy / len(dataloader)

# Main function

In [9]:
def parse_args():
  """arguments"""
  config = {
    "data_dir": "./Dataset",
    "save_path": "model.ckpt",
    "batch_size": 64,
    "n_workers": 2,
    "valid_steps": 2000,
    "warmup_steps": 1000,
    "save_steps": 10000,
    "total_steps": 70000,
  }

  return config


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def main(
  data_dir,
  save_path,
  batch_size,
  n_workers,
  valid_steps,
  warmup_steps,
  total_steps,
  save_steps,
):
  """Main function."""
  device = get_device()
  print(f"[Info]: Use {device} now!")

  train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers)
  train_iterator = iter(train_loader)
  print(f"[Info]: Finish loading data!",flush = True)

  model = Classifier(n_spks=speaker_num).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = AdamW(model.parameters(), lr=1e-3)
  scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
  print(f"[Info]: Finish creating model!",flush = True)

  best_accuracy = -1.0
  best_state_dict = None

  pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

  for step in range(total_steps):
    # Get data
    try:
      batch = next(train_iterator)
    except StopIteration:
      train_iterator = iter(train_loader)
      batch = next(train_iterator)

    loss, accuracy = model_fn(batch, model, criterion, device)
    batch_loss = loss.item()
    batch_accuracy = accuracy.item()

    # Updata model
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    
    # Log
    pbar.update()
    pbar.set_postfix(
      loss=f"{batch_loss:.2f}",
      accuracy=f"{batch_accuracy:.2f}",
      step=step + 1,
    )

    # Do validation
    if (step + 1) % valid_steps == 0:
      pbar.close()

      valid_accuracy = valid(valid_loader, model, criterion, device)

      # keep the best model
      if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy
        best_state_dict = model.state_dict()

      pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

    # Save the best model so far.
    if (step + 1) % save_steps == 0 and best_state_dict is not None:
      torch.save(best_state_dict, save_path)
      pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

  pbar.close()


if __name__ == "__main__":
  main(**parse_args())

[Info]: Use mps now!


FileNotFoundError: [Errno 2] No such file or directory: 'Dataset/mapping.json'

# Inference

## Dataset of inference

In [ ]:
class InferenceDataset(Dataset):
  def __init__(self, data_dir):
    testdata_path = Path(data_dir) / "testdata.json"
    metadata = json.load(testdata_path.open())
    self.data_dir = data_dir
    self.data = metadata["utterances"]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    utterance = self.data[index]
    feat_path = utterance["feature_path"]
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    return feat_path, mel


def inference_collate_batch(batch):
  """Collate a batch of data."""
  feat_paths, mels = zip(*batch)

  return feat_paths, torch.stack(mels)

## Main funcrion of Inference

In [ ]:
def parse_args():
  """arguments"""
  config = {
    "data_dir": "./Dataset",
    "model_path": "./model.ckpt",
    "output_path": "./output.csv",
  }

  return config


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def main(
  data_dir,
  model_path,
  output_path,
):
  """Main function."""
  device = get_device()
  print(f"[Info]: Use {device} now!")

  mapping_path = Path(data_dir) / "mapping.json"
  mapping = json.load(mapping_path.open())

  dataset = InferenceDataset(data_dir)
  dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    drop_last=False,
    num_workers=8,
    collate_fn=inference_collate_batch,
  )
  print(f"[Info]: Finish loading data!",flush = True)

  speaker_num = len(mapping["id2speaker"])
  model = Classifier(n_spks=speaker_num).to(device)
  model.load_state_dict(torch.load(model_path))
  model.eval()
  print(f"[Info]: Finish creating model!",flush = True)

  results = [["Id", "Category"]]
  for feat_paths, mels in notebook_tqdm(dataloader):
    with torch.no_grad():
      mels = mels.to(device)
      outs = model(mels)
      preds = outs.argmax(1).cpu().numpy()
      for feat_path, pred in zip(feat_paths, preds):
        results.append([feat_path, mapping["id2speaker"][str(pred)]])
  
  with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(results)


if __name__ == "__main__":
  main(**parse_args())